In [4]:
!pip install gradio
import gradio as gr
import tensorflow as tf
import numpy as np
from PIL import Image

In [5]:
# Ensure your model file is in the same folder as this notebook
model = tf.keras.models.load_model('fish_classifier_model.h5')

In [6]:
# Replace these with your actual 5 fish species names in the correct order
class_names = ['Anthias anthias', 'Belone belone', 'Coris julis', 'Dasyatis centroura', 'Scomber japonicus']

def predict_fish(img):
    # Resize image to match model input shape
    img = img.resize((150, 150))
    # Convert to array and normalize (0-1 range)
    img_array = np.array(img) / 255.0
    # Add a batch dimension (shape becomes 1, 150, 150, 3)
    img_array = np.expand_dims(img_array, axis=0)

    # Make prediction
    prediction = model.predict(img_array)[0]

    # Return a dictionary of {Species: Probability} for the GUI to display
    return {class_names[i]: float(prediction[i]) for i in range(len(class_names))}

In [7]:
import gradio as gr

# assume you already have:
# def predict_fish(image): ...

THEME = gr.themes.Soft(
    primary_hue="blue",
    secondary_hue="cyan",
    neutral_hue="slate",
).set(
    body_background_fill="*neutral_50",
    block_background_fill="white",
    block_border_width="1px",
    block_shadow="*shadow_sm",
    button_primary_background_fill="*primary_600",
    button_primary_background_fill_hover="*primary_700",
)

CSS = """
#title { text-align: center; }
#subtitle { text-align: center; margin-top: -8px; opacity: 0.85; }
.card { border-radius: 16px; }
.hint { font-size: 0.92rem; opacity: 0.85; }
.footer { font-size: 0.9rem; opacity: 0.75; text-align: center; }
"""

with gr.Blocks(theme=THEME, css=CSS, title="Fish Species Identification") as app:

    gr.Markdown(
        """
        # 🐟 Fish Species Identification System
        <div id="subtitle">CNN-based Image Classification Demo • Upload a fish image to predict its species</div>
        """,
        elem_id="title",
    )

    with gr.Row(equal_height=True):
        # LEFT: Inputs
        with gr.Column(scale=6, min_width=360):
            with gr.Group(elem_classes=["card"]):
                gr.Markdown("### 📤 Input", elem_classes=["hint"])
                image_input = gr.Image(
                    type="pil",
                    label="Upload Fish Image",
                    height=320,
                    sources=["upload", "webcam"],  # allow webcam too
                )

                gr.Markdown(
                    "Tip: Use a clear, well-lit image where the fish body is visible (not too far/blurred).",
                    elem_classes=["hint"],
                )

                with gr.Row():
                    submit_btn = gr.Button("🔍 Identify", variant="primary")
                    clear_btn = gr.Button("🧹 Clear", variant="secondary")

                status = gr.Markdown("", elem_classes=["hint"])

        # RIGHT: Outputs
        with gr.Column(scale=6, min_width=360):
            with gr.Group(elem_classes=["card"]):
                gr.Markdown("### 📊 Output", elem_classes=["hint"])

                output_label = gr.Label(
                    num_top_classes=3,
                    label="Prediction Results (Top-3)",
                )

                with gr.Accordion("How to read the results", open=False):
                    gr.Markdown(
                        """
                        - The top label is the model's best guess.
                        - The scores reflect confidence (higher is better).
                        - If confidence is low, try another photo (better lighting/angle).
                        """,
                        elem_classes=["hint"],
                    )

    with gr.Row():
        with gr.Accordion("📚 Project Info", open=False):
            gr.Markdown(
                """
                **Model:** Convolutional Neural Network (CNN)
                **Course:** CSC583 – Image Classification Project
                **Note:** Results depend on image quality and lighting.
                """,
            )

    # --- Events ---
    def _predict_with_status(img):
        if img is None:
            return gr.update(value={"error": "Please upload an image first."}), "⚠️ No image provided."
        # call your real predictor
        preds = predict_fish(img)
        return preds, "✅ Done. You can try another image."

    submit_btn.click(
        fn=_predict_with_status,
        inputs=image_input,
        outputs=[output_label, status],
    )

    clear_btn.click(
        fn=lambda: (None, None, ""),
        inputs=[],
        outputs=[image_input, output_label, status],
    )

app.queue()  # smoother for multiple users / slower models
app.launch(share=True)


/tmp/ipython-input-2341147298.py:27: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=THEME, css=CSS, title="Fish Species Identification") as app:
/tmp/ipython-input-2341147298.py:27: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(theme=THEME, css=CSS, title="Fish Species Identification") as app:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://955191dc223e0d8b4b.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
